# Environmental Disclosure Checklist Items

Retrieve and analyze all Checklist Items from the configuration and DuckDB database.

This notebook:
1. Loads checklist items from `config.py`
2. Queries evaluation results from DuckDB if available
3. Displays, filters, and exports checklist data

## 1. Import Required Libraries

In [6]:
from pathlib import Path
import sys
import json

import duckdb
import pandas as pd

# Add project root to path for imports
project_root = Path('/media/nvme0n1/dev/annual_report')
sys.path.insert(0, str(project_root))

from config import CHECKLIST_ITEMS, DB_PATH

print("Libraries imported successfully")
print(f"Project root: {project_root}")
print(f"Database path: {DB_PATH}")

Libraries imported successfully
Project root: /media/nvme0n1/dev/annual_report
Database path: /media/nvme0n1/dev/annual_report/db.db


## 2. Connect to DuckDB Database

In [7]:
# Connect to DuckDB database
db_path = Path(DB_PATH)
assert db_path.exists(), f'Database file not found: {db_path}'

con = duckdb.connect(str(db_path), read_only=True)
con.execute('PRAGMA threads=4;')

print(f"✓ Connected to: {db_path}")
print(f"✓ Database info: {con.execute('SELECT current_database(), current_schema()').fetchall()}")

✓ Connected to: /media/nvme0n1/dev/annual_report/db.db
✓ Database info: [('db', 'main')]


## 3. Load Checklist Items from Configuration

In [3]:
# Convert checklist items to DataFrame
checklist_df = pd.DataFrame(CHECKLIST_ITEMS)

print(f"✓ Loaded {len(checklist_df)} checklist items")
print(f"\nColumns: {list(checklist_df.columns)}")
print(f"Data types:\n{checklist_df.dtypes}")
print(f"\nChecklist Items Summary:")
print(checklist_df)

✓ Loaded 18 checklist items

Columns: ['code', 'group', 'item', 'description']
Data types:
code           str
group          str
item           str
description    str
dtype: object

Checklist Items Summary:
    code               group  \
0    CC1      Climate Change   
1    CC2      Climate Change   
2   GHG1       GHG Emissions   
3   GHG2       GHG Emissions   
4   GHG3       GHG Emissions   
5   GHG4       GHG Emissions   
6   GHG5       GHG Emissions   
7   GHG6       GHG Emissions   
8   GHG7       GHG Emissions   
9    EC1  Energy Consumption   
10   EC2  Energy Consumption   
11   EC3  Energy Consumption   
12   RC1       GHG Reduction   
13   RC2       GHG Reduction   
14   RC3       GHG Reduction   
15   RC4       GHG Reduction   
16  ACC1  GHG Accountability   
17  ACC2  GHG Accountability   

                                                 item  \
0   Assessment of climate-related risks and opport...   
1            Financial implications of climate change   
2            

## 4. Query Checklist Evaluation Results from DuckDB

In [4]:
# Check for inference results table
try:
    # Query inference results
    inference_results_df = con.execute("""
        SELECT 
            ticker,
            year,
            category_code,
            is_valid,
            reason,
            model,
            created_at
        FROM inference_results
        ORDER BY ticker, year, category_code
        LIMIT 1000
    """).df()
    
    print(f"✓ Found {len(inference_results_df)} inference results")
    print(f"\nInference Results Summary:")
    print(inference_results_df.head(20))
    
except Exception as e:
    print(f"⚠ No inference results table found or error querying: {e}")
    inference_results_df = None

✓ Found 1000 inference results

Inference Results Summary:
   ticker  year category_code  is_valid  \
0     AAA  2015          ACC1     False   
1     AAA  2015      ACC1_alt     False   
2     AAA  2015  ACC1_alt_two      True   
3     AAA  2015          ACC2     False   
4     AAA  2015      ACC2_alt     False   
5     AAA  2015  ACC2_alt_two     False   
6     AAA  2015           CC1     False   
7     AAA  2015       CC1_alt      True   
8     AAA  2015   CC1_alt_two      True   
9     AAA  2015           CC2     False   
10    AAA  2015       CC2_alt      True   
11    AAA  2015   CC2_alt_two      True   
12    AAA  2015           EC1     False   
13    AAA  2015       EC1_alt      True   
14    AAA  2015   EC1_alt_two      True   
15    AAA  2015           EC2     False   
16    AAA  2015       EC2_alt     False   
17    AAA  2015   EC2_alt_two     False   
18    AAA  2015           EC3     False   
19    AAA  2015       EC3_alt      True   

                                     

## 5. Display and Inspect Checklist Items Details

In [5]:
# Display checklist items grouped by category
print("=" * 80)
print("CHECKLIST ITEMS BY GROUP")
print("=" * 80)

for group_name in checklist_df['group'].unique():
    group_items = checklist_df[checklist_df['group'] == group_name]
    print(f"\n{group_name} ({len(group_items)} items):")
    print("-" * 80)
    for idx, row in group_items.iterrows():
        print(f"  [{row['code']}] {row['item']}")
        print(f"       Description (VN): {row['description'][:80]}...")
        print()

CHECKLIST ITEMS BY GROUP

Climate Change (2 items):
--------------------------------------------------------------------------------
  [CC1] Assessment of climate-related risks and opportunities
       Description (VN): Đánh giá các rủi ro (các quy định, tác động vật lý hoặc các tác động chung) liên...

  [CC2] Financial implications of climate change
       Description (VN): Đánh giá các tác động tài chính hiện tại (và tương lai), tác động kinh doanh và ...


GHG Emissions (7 items):
--------------------------------------------------------------------------------
  [GHG1] Methodology for GHG emission calculation
       Description (VN): Mô tả các phương pháp sử dụng để tính toán khí thải nhà kính....

  [GHG2] External verification of GHG emissions
       Description (VN): Có sự xác nhận bởi yếu tố bên ngoài về lượng phát thải khí nhà kính hay không? N...

  [GHG3] Total GHG emissions disclosed
       Description (VN): Lượng phát thải khí nhà kính tính bằng đơn vị MtCO2e (hệ mét tấn C

## 6. Filter and Analyze Checklist Items

In [6]:
# Statistics by group
print("\n" + "=" * 80)
print("CHECKLIST STATISTICS")
print("=" * 80)

group_stats = checklist_df.groupby('group').size().reset_index(name='count')
print("\nItems per Group:")
print(group_stats)
print(f"\nTotal Checklist Items: {len(checklist_df)}")

# Example: Filter by specific group
print("\n" + "-" * 80)
print("Example: Climate Change Items Only")
print("-" * 80)
cc_items = checklist_df[checklist_df['group'] == 'Climate Change']
print(cc_items[['code', 'item', 'group']].to_string(index=False))


CHECKLIST STATISTICS

Items per Group:
                group  count
0      Climate Change      2
1  Energy Consumption      3
2  GHG Accountability      2
3       GHG Emissions      7
4       GHG Reduction      4

Total Checklist Items: 18

--------------------------------------------------------------------------------
Example: Climate Change Items Only
--------------------------------------------------------------------------------
code                                                  item          group
 CC1 Assessment of climate-related risks and opportunities Climate Change
 CC2              Financial implications of climate change Climate Change


## 7. Merge Checklist Items with Evaluation Results (if available)

In [7]:
# Merge checklist items with evaluation results
if inference_results_df is not None and len(inference_results_df) > 0:
    # Rename category_code to code for merge
    inference_df_rename = inference_results_df.rename(columns={'category_code': 'code'})
    
    # Merge
    merged_df = pd.merge(checklist_df, inference_df_rename, on='code', how='left')
    
    print(f"✓ Merged checklist items with {len(inference_results_df)} evaluation results")
    print(f"\nMerged Data Sample:")
    print(merged_df[['code', 'group', 'item', 'ticker', 'year', 'is_valid', 'model']].head(20))
    
    # Summary stats
    if 'is_valid' in merged_df.columns:
        print("\n" + "-" * 80)
        print("Evaluation Results Summary:")
        valid_counts = merged_df['is_valid'].value_counts()
        print(f"  Valid: {valid_counts.get(True, 0)}")
        print(f"  Invalid: {valid_counts.get(False, 0)}")
else:
    print("⚠ No evaluation results to merge")

✓ Merged checklist items with 1000 evaluation results

Merged Data Sample:
   code           group                                               item  \
0   CC1  Climate Change  Assessment of climate-related risks and opport...   
1   CC1  Climate Change  Assessment of climate-related risks and opport...   
2   CC1  Climate Change  Assessment of climate-related risks and opport...   
3   CC1  Climate Change  Assessment of climate-related risks and opport...   
4   CC1  Climate Change  Assessment of climate-related risks and opport...   
5   CC1  Climate Change  Assessment of climate-related risks and opport...   
6   CC1  Climate Change  Assessment of climate-related risks and opport...   
7   CC1  Climate Change  Assessment of climate-related risks and opport...   
8   CC1  Climate Change  Assessment of climate-related risks and opport...   
9   CC1  Climate Change  Assessment of climate-related risks and opport...   
10  CC1  Climate Change  Assessment of climate-related risks and op

## 8. Export Results to Various Formats

In [8]:
# Export checklist items to various formats
output_dir = Path('/media/nvme0n1/dev/annual_report/data/output')
output_dir.mkdir(parents=True, exist_ok=True)

# Export to CSV
csv_path = output_dir / 'checklist_items.csv'
checklist_df.to_csv(csv_path, index=False)
print(f"✓ Exported to CSV: {csv_path}")

# Export to JSON
json_path = output_dir / 'checklist_items.json'
checklist_df.to_json(json_path, orient='records', indent=2)
print(f"✓ Exported to JSON: {json_path}")

# Export to Parquet
parquet_path = output_dir / 'checklist_items.parquet'
checklist_df.to_parquet(parquet_path, index=False)
print(f"✓ Exported to Parquet: {parquet_path}")

# Export to Excel (if available)
try:
    excel_path = output_dir / 'checklist_items.xlsx'
    checklist_df.to_excel(excel_path, index=False, sheet_name='Checklist Items')
    print(f"✓ Exported to Excel: {excel_path}")
except ImportError:
    print("⚠ openpyxl not installed, skipping Excel export")

print(f"\n✓ All exports completed to: {output_dir}")

✓ Exported to CSV: /media/nvme0n1/dev/annual_report/data/output/checklist_items.csv
✓ Exported to JSON: /media/nvme0n1/dev/annual_report/data/output/checklist_items.json
✓ Exported to Parquet: /media/nvme0n1/dev/annual_report/data/output/checklist_items.parquet
⚠ openpyxl not installed, skipping Excel export

✓ All exports completed to: /media/nvme0n1/dev/annual_report/data/output


## 9. Get ALL Inference Results (Complete Dataset)

In [10]:
# Get complete count of inference results
try:
    total_count = con.execute("SELECT COUNT(*) as count FROM inference_results").fetchone()[0]
    print(f"✓ Total inference results in database: {total_count:,}")
    
    # Query ALL inference results
    all_results = con.execute("""
        SELECT 
            ticker,
            year,
            category_code,
            is_valid,
            reason,
            top_chunks,
            similarities,
            model,
            created_at
        FROM inference_results
        ORDER BY ticker, year, category_code
    """).df()
    
    print(f"✓ Retrieved {len(all_results):,} inference results")
    print(f"\nDataframe Info:")
    print(f"  Shape: {all_results.shape}")
    print(f"  Columns: {list(all_results.columns)}")
    print(f"  Memory usage: {all_results.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    
except Exception as e:
    print(f"✗ Error querying inference results: {e}")
    all_results = None

✓ Total inference results in database: 50,058
✓ Retrieved 50,058 inference results

Dataframe Info:
  Shape: (50058, 9)
  Columns: ['ticker', 'year', 'category_code', 'is_valid', 'reason', 'top_chunks', 'similarities', 'model', 'created_at']
  Memory usage: 26.13 MB


## 10. Detailed Inference Results Statistics

In [10]:
if all_results is not None and len(all_results) > 0:
    print("=" * 100)
    print("INFERENCE RESULTS ANALYSIS")
    print("=" * 100)
    
    # Overall statistics
    print("\n1. OVERALL STATISTICS:")
    print("-" * 100)
    total_valid = all_results['is_valid'].sum()
    total_invalid = (~all_results['is_valid']).sum()
    valid_pct = (total_valid / len(all_results)) * 100
    print(f"  Total Results: {len(all_results):,}")
    print(f"  Valid (True):  {total_valid:,} ({valid_pct:.1f}%)")
    print(f"  Invalid (False): {total_invalid:,} ({100-valid_pct:.1f}%)")
    
    # By ticker
    print("\n2. RESULTS BY TICKER (Top 20):")
    print("-" * 100)
    ticker_stats = all_results.groupby('ticker').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    ticker_stats.columns = ['Valid', 'Total', 'Valid %']
    ticker_stats = ticker_stats.sort_values('Total', ascending=False).head(20)
    print(ticker_stats)
    
    # By year
    print("\n3. RESULTS BY YEAR:")
    print("-" * 100)
    year_stats = all_results.groupby('year').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    year_stats.columns = ['Valid', 'Total', 'Valid %']
    year_stats = year_stats.sort_index()
    print(year_stats)
    
    # By category code
    print("\n4. RESULTS BY CATEGORY CODE (All):")
    print("-" * 100)
    category_stats = all_results.groupby('category_code').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    category_stats.columns = ['Valid', 'Total', 'Valid %']
    category_stats = category_stats.sort_values('Total', ascending=False)
    print(category_stats)
    
    # By model
    print("\n5. RESULTS BY MODEL:")
    print("-" * 100)
    model_stats = all_results.groupby('model').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    model_stats.columns = ['Valid', 'Total', 'Valid %']
    print(model_stats)
    
else:
    print("No inference results found")

INFERENCE RESULTS ANALYSIS

1. OVERALL STATISTICS:
----------------------------------------------------------------------------------------------------
  Total Results: 50,058
  Valid (True):  16,930 (33.8%)
  Invalid (False): 33,128 (66.2%)

2. RESULTS BY TICKER (Top 20):
----------------------------------------------------------------------------------------------------
        Valid  Total  Valid %
ticker                       
AAA       297    594     50.0
ACB       173    594     29.1
ARM       156    594     26.3
C32       255    594     42.9
CDN       157    594     26.4
CCR       225    594     37.9
CAG       204    594     34.3
GSP       207    594     34.8
FPT       278    594     46.8
GEG       373    594     62.8
CLL       190    594     32.0
DHG       169    594     28.5
CNG       280    594     47.1
DL1       202    594     34.0
DVP       176    594     29.6
GMD       301    594     50.7
GMC       274    594     46.1
HTV       147    594     24.7
HMH        22    594     

## 11. View Sample Inference Results

In [11]:
if all_results is not None and len(all_results) > 0:
    print("=" * 100)
    print("SAMPLE DATA")
    print("=" * 100)
    
    # Display first few rows
    print("\nFirst 10 Results:")
    print("-" * 100)
    display_cols = ['ticker', 'year', 'category_code', 'is_valid', 'model']
    print(all_results[display_cols].head(10).to_string(index=False))
    
    # Show a sample valid result with reason
    valid_results = all_results[all_results['is_valid'] == True]
    if len(valid_results) > 0:
        print("\n\nSample VALID Result (with reason):")
        print("-" * 100)
        sample_valid = valid_results.iloc[0]
        print(f"Ticker: {sample_valid['ticker']}")
        print(f"Year: {sample_valid['year']}")
        print(f"Category: {sample_valid['category_code']}")
        print(f"Valid: {sample_valid['is_valid']}")
        print(f"Model: {sample_valid['model']}")
        print(f"Reason: {sample_valid['reason']}")
    
    # Show a sample invalid result with reason
    invalid_results = all_results[all_results['is_valid'] == False]
    if len(invalid_results) > 0:
        print("\n\nSample INVALID Result (with reason):")
        print("-" * 100)
        sample_invalid = invalid_results.iloc[0]
        print(f"Ticker: {sample_invalid['ticker']}")
        print(f"Year: {sample_invalid['year']}")
        print(f"Category: {sample_invalid['category_code']}")
        print(f"Valid: {sample_invalid['is_valid']}")
        print(f"Model: {sample_invalid['model']}")
        print(f"Reason: {sample_invalid['reason']}")
else:
    print("No results to display")

SAMPLE DATA

First 10 Results:
----------------------------------------------------------------------------------------------------
ticker  year category_code  is_valid        model
   AAA  2015          ACC1     False gpt-4.1-mini
   AAA  2015      ACC1_alt     False gpt-4.1-mini
   AAA  2015  ACC1_alt_two      True gpt-4.1-mini
   AAA  2015          ACC2     False gpt-4.1-mini
   AAA  2015      ACC2_alt     False gpt-4.1-mini
   AAA  2015  ACC2_alt_two     False gpt-4.1-mini
   AAA  2015           CC1     False gpt-4.1-mini
   AAA  2015       CC1_alt      True gpt-4.1-mini
   AAA  2015   CC1_alt_two      True gpt-4.1-mini
   AAA  2015           CC2     False gpt-4.1-mini


Sample VALID Result (with reason):
----------------------------------------------------------------------------------------------------
Ticker: AAA
Year: 2015
Category: ACC1_alt_two
Valid: True
Model: gpt-4.1-mini
Reason: The text clearly identifies the company's management and supervisory bodies, such as the Board

## 12. Export ALL Inference Results

In [12]:
if all_results is not None and len(all_results) > 0:
    output_dir = Path('/media/nvme0n1/dev/annual_report/data/output')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Export to CSV
    csv_path = output_dir / 'all_inference_results.csv'
    all_results.to_csv(csv_path, index=False)
    print(f"✓ Exported to CSV: {csv_path}")
    print(f"  Size: {csv_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to JSON
    json_path = output_dir / 'all_inference_results.json'
    all_results.to_json(json_path, orient='records', indent=2)
    print(f"✓ Exported to JSON: {json_path}")
    print(f"  Size: {json_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to Parquet
    parquet_path = output_dir / 'all_inference_results.parquet'
    all_results.to_parquet(parquet_path, index=False)
    print(f"✓ Exported to Parquet: {parquet_path}")
    print(f"  Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export statistics to CSV
    stats_path = output_dir / 'inference_results_summary_stats.csv'
    summary_stats = pd.DataFrame({
        'Metric': ['Total Results', 'Valid Count', 'Invalid Count', 'Valid %', 'Unique Tickers', 'Unique Years', 'Unique Categories', 'Unique Models'],
        'Value': [
            len(all_results),
            all_results['is_valid'].sum(),
            (~all_results['is_valid']).sum(),
            f"{(all_results['is_valid'].sum() / len(all_results) * 100):.1f}%",
            all_results['ticker'].nunique(),
            all_results['year'].nunique(),
            all_results['category_code'].nunique(),
            all_results['model'].nunique()
        ]
    })
    summary_stats.to_csv(stats_path, index=False)
    print(f"✓ Exported summary stats to CSV: {stats_path}")
    
    print(f"\n✓ All exports completed to: {output_dir}")
else:
    print("No inference results to export")

✓ Exported to CSV: /media/nvme0n1/dev/annual_report/data/output/all_inference_results.csv
  Size: 25.56 MB
✓ Exported to JSON: /media/nvme0n1/dev/annual_report/data/output/all_inference_results.json
  Size: 32.63 MB
✓ Exported to Parquet: /media/nvme0n1/dev/annual_report/data/output/all_inference_results.parquet
  Size: 9.96 MB
✓ Exported summary stats to CSV: /media/nvme0n1/dev/annual_report/data/output/inference_results_summary_stats.csv

✓ All exports completed to: /media/nvme0n1/dev/annual_report/data/output


/tmp/ipykernel_57519/1163659763.py:13: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  all_results.to_json(json_path, orient='records', indent=2)


## 13. Get HyDE2 Inference Results

In [8]:
# Query HyDE2 inference results
try:
    hyde2_count = con.execute("SELECT COUNT(*) as count FROM inference_results_hyde2").fetchone()[0]
    print(f"✓ Total HyDE2 inference results in database: {hyde2_count:,}")
    
    # Query ALL HyDE2 results
    hyde2_results = con.execute("""
        SELECT 
            ticker,
            year,
            category_code,
            is_valid,
            reason,
            top_chunks,
            similarities,
            model,
            created_at
        FROM inference_results_hyde2
        ORDER BY ticker, year, category_code
    """).df()
    
    print(f"✓ Retrieved {len(hyde2_results):,} HyDE2 inference results")
    print(f"\nDataframe Info:")
    print(f"  Shape: {hyde2_results.shape}")
    print(f"  Columns: {list(hyde2_results.columns)}")
    print(f"  Memory usage: {hyde2_results.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    
except Exception as e:
    print(f"✗ Error querying HyDE2 results: {e}")
    hyde2_results = None

✓ Total HyDE2 inference results in database: 33,372
✓ Retrieved 33,372 HyDE2 inference results

Dataframe Info:
  Shape: (33372, 9)
  Columns: ['ticker', 'year', 'category_code', 'is_valid', 'reason', 'top_chunks', 'similarities', 'model', 'created_at']
  Memory usage: 21.27 MB


## 14. Filter EDC Results from HyDE2

In [9]:
if hyde2_results is not None and len(hyde2_results) > 0:
    # EDC base category codes
    edc_codes = ['CC1', 'CC2', 'GHG1', 'GHG2', 'GHG3', 'GHG4', 'GHG5', 'GHG6', 'GHG7', 
                 'EC1', 'EC2', 'EC3', 'RC1', 'RC2', 'RC3', 'RC4', 'ACC1', 'ACC2']
    
    # Filter for all EDC versions (base, alt, alt_two)
    hyde2_edc = hyde2_results[
        hyde2_results['category_code'].str.split('_').str[0].isin(edc_codes)
    ].copy()
    
    print(f"✓ Filtered {len(hyde2_edc):,} EDC-related HyDE2 results")
    print(f"  Base EDC codes found: {hyde2_edc['category_code'].nunique()}")
    
    # Separate into standard, alt, and alt_two
    hyde2_standard = hyde2_edc[~hyde2_edc['category_code'].str.contains('_alt', regex=True)]
    hyde2_alt = hyde2_edc[hyde2_edc['category_code'].str.contains('_alt_two', regex=True)]
    hyde2_alt_two = hyde2_edc[hyde2_edc['category_code'].str.contains('_alt_two', regex=True)]
    
    print(f"\nBreakdown by version:")
    print(f"  Standard EDC: {len(hyde2_standard):,} results")
    print(f"  Alternative (_alt): {len(hyde2_alt):,} results")
    print(f"  Alternative 2 (_alt_two): {len(hyde2_alt_two):,} results")
    
    print(f"\nEDC Versions in HyDE2 Results:")
    print("-" * 80)
    version_counts = hyde2_edc['category_code'].value_counts().sort_index()
    print(version_counts)
    
else:
    print("No HyDE2 results to filter")
    hyde2_edc = None

✓ Filtered 33,372 EDC-related HyDE2 results
  Base EDC codes found: 36

Breakdown by version:
  Standard EDC: 0 results
  Alternative (_alt): 16,686 results
  Alternative 2 (_alt_two): 16,686 results

EDC Versions in HyDE2 Results:
--------------------------------------------------------------------------------
category_code
ACC1_alt        927
ACC1_alt_two    927
ACC2_alt        927
ACC2_alt_two    927
CC1_alt         927
CC1_alt_two     927
CC2_alt         927
CC2_alt_two     927
EC1_alt         927
EC1_alt_two     927
EC2_alt         927
EC2_alt_two     927
EC3_alt         927
EC3_alt_two     927
GHG1_alt        927
GHG1_alt_two    927
GHG2_alt        927
GHG2_alt_two    927
GHG3_alt        927
GHG3_alt_two    927
GHG4_alt        927
GHG4_alt_two    927
GHG5_alt        927
GHG5_alt_two    927
GHG6_alt        927
GHG6_alt_two    927
GHG7_alt        927
GHG7_alt_two    927
RC1_alt         927
RC1_alt_two     927
RC2_alt         927
RC2_alt_two     927
RC3_alt         927
RC3_alt_two  

## 15. EDC HyDE2 Results Statistics

In [8]:
if hyde2_edc is not None and len(hyde2_edc) > 0:
    print("=" * 100)
    print("EDC HyDE2 RESULTS ANALYSIS")
    print("=" * 100)
    
    # Overall statistics
    print("\n1. OVERALL STATISTICS:")
    print("-" * 100)
    total_valid = hyde2_edc['is_valid'].sum()
    total_invalid = (~hyde2_edc['is_valid']).sum()
    valid_pct = (total_valid / len(hyde2_edc)) * 100
    print(f"  Total EDC Results: {len(hyde2_edc):,}")
    print(f"  Valid (True):  {total_valid:,} ({valid_pct:.1f}%)")
    print(f"  Invalid (False): {total_invalid:,} ({100-valid_pct:.1f}%)")
    
    # By category code
    print("\n2. RESULTS BY EDC CATEGORY CODE:")
    print("-" * 100)
    category_stats = hyde2_edc.groupby('category_code').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    category_stats.columns = ['Valid', 'Total', 'Valid %']
    category_stats = category_stats.sort_values('Total', ascending=False)
    print(category_stats)
    
    # By ticker (top 15)
    print("\n3. RESULTS BY TICKER (Top 15):")
    print("-" * 100)
    ticker_stats = hyde2_edc.groupby('ticker').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    ticker_stats.columns = ['Valid', 'Total', 'Valid %']
    ticker_stats = ticker_stats.sort_values('Total', ascending=False).head(15)
    print(ticker_stats)
    
    # By year
    print("\n4. RESULTS BY YEAR:")
    print("-" * 100)
    year_stats = hyde2_edc.groupby('year').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    year_stats.columns = ['Valid', 'Total', 'Valid %']
    year_stats = year_stats.sort_index()
    print(year_stats)
    
    # By version (standard, alt, alt_two)
    print("\n5. RESULTS BY VERSION:")
    print("-" * 100)
    hyde2_edc['version'] = hyde2_edc['category_code'].apply(
        lambda x: 'alt_two' if '_alt_two' in x else ('alt' if '_alt' in x else 'standard')
    )
    version_stats = hyde2_edc.groupby('version').agg({
        'is_valid': ['sum', 'count', lambda x: (x.sum() / len(x) * 100).round(1)]
    }).round(1)
    version_stats.columns = ['Valid', 'Total', 'Valid %']
    print(version_stats)
    
else:
    print("No EDC HyDE2 results to analyze")

EDC HyDE2 RESULTS ANALYSIS

1. OVERALL STATISTICS:
----------------------------------------------------------------------------------------------------
  Total EDC Results: 33,372
  Valid (True):  16,545 (49.6%)
  Invalid (False): 16,827 (50.4%)

2. RESULTS BY EDC CATEGORY CODE:
----------------------------------------------------------------------------------------------------
               Valid  Total  Valid %
category_code                       
ACC1_alt         489    927     52.8
ACC1_alt_two     632    927     68.2
ACC2_alt         375    927     40.5
ACC2_alt_two      84    927      9.1
CC1_alt          596    927     64.3
CC1_alt_two      657    927     70.9
CC2_alt          489    927     52.8
CC2_alt_two      657    927     70.9
EC1_alt          603    927     65.0
EC1_alt_two      482    927     52.0
EC2_alt          360    927     38.8
EC2_alt_two      205    927     22.1
EC3_alt          544    927     58.7
EC3_alt_two      435    927     46.9
GHG1_alt         713    927

## 16. Compare Standard vs HyDE2 EDC Results

In [11]:
if all_results is not None and hyde2_edc is not None:
    # Filter all_results for EDC items (standard versions only)
    edc_codes = ['CC1', 'CC2', 'GHG1', 'GHG2', 'GHG3', 'GHG4', 'GHG5', 'GHG6', 'GHG7', 
                 'EC1', 'EC2', 'EC3', 'RC1', 'RC2', 'RC3', 'RC4', 'ACC1', 'ACC2']
    
    standard_edc = all_results[all_results['category_code'].isin(edc_codes)].copy()
    
    print("=" * 100)
    print("STANDARD vs HyDE2 EDC RESULTS COMPARISON")
    print("=" * 100)
    
    print("\n1. OVERALL COMPARISON:")
    print("-" * 100)
    print(f"  Standard EDC Results: {len(standard_edc):,}")
    print(f"    - Valid: {standard_edc['is_valid'].sum():,} ({(standard_edc['is_valid'].sum() / len(standard_edc) * 100):.1f}%)")
    print(f"\n  HyDE2 EDC Results: {len(hyde2_edc):,}")
    print(f"    - Valid: {hyde2_edc['is_valid'].sum():,} ({(hyde2_edc['is_valid'].sum() / len(hyde2_edc) * 100):.1f}%)")
    
    # Compare by category code (standard versions only)
    print("\n2. BY CATEGORY CODE (Standard versions):")
    print("-" * 100)
    comp_cats = list(set(standard_edc['category_code'].unique()) & set(hyde2_edc['category_code'].unique()))
    comp_cats.sort()
    
    comparison_data = []
    for cat in comp_cats:
        std_results = standard_edc[standard_edc['category_code'] == cat]
        hyde2_cat_results = hyde2_edc[hyde2_edc['category_code'] == cat]
        
        if len(std_results) > 0 or len(hyde2_cat_results) > 0:
            std_valid_pct = (std_results['is_valid'].sum() / len(std_results) * 100) if len(std_results) > 0 else 0
            hyde2_valid_pct = (hyde2_cat_results['is_valid'].sum() / len(hyde2_cat_results) * 100) if len(hyde2_cat_results) > 0 else 0
            
            comparison_data.append({
                'Category': cat,
                'Std Valid %': f"{std_valid_pct:.1f}%",
                'HyDE2 Valid %': f"{hyde2_valid_pct:.1f}%",
                'Difference': f"{(hyde2_valid_pct - std_valid_pct):+.1f}%"
            })
    
    if comparison_data:
        comp_df = pd.DataFrame(comparison_data)
        print(comp_df.to_string(index=False))
    
    print("\n3. TOP 10 TICKERS COMPARISON:")
    print("-" * 100)
    top_tickers = standard_edc['ticker'].value_counts().head(10).index.tolist()
    
    ticker_comp = []
    for ticker in top_tickers:
        std_t = standard_edc[standard_edc['ticker'] == ticker]
        hyde2_t = hyde2_edc[hyde2_edc['ticker'] == ticker]
        
        std_valid_pct = (std_t['is_valid'].sum() / len(std_t) * 100) if len(std_t) > 0 else 0
        hyde2_valid_pct = (hyde2_t['is_valid'].sum() / len(hyde2_t) * 100) if len(hyde2_t) > 0 else 0
        
        ticker_comp.append({
            'Ticker': ticker,
            'Std Valid %': f"{std_valid_pct:.1f}%",
            'HyDE2 Valid %': f"{hyde2_valid_pct:.1f}%",
            'Difference': f"{(hyde2_valid_pct - std_valid_pct):+.1f}%"
        })
    
    ticker_comp_df = pd.DataFrame(ticker_comp)
    print(ticker_comp_df.to_string(index=False))
    
else:
    print("Cannot compare - missing standard or HyDE2 results")

STANDARD vs HyDE2 EDC RESULTS COMPARISON

1. OVERALL COMPARISON:
----------------------------------------------------------------------------------------------------
  Standard EDC Results: 16,686
    - Valid: 385 (2.3%)

  HyDE2 EDC Results: 33,372
    - Valid: 16,545 (49.6%)

2. BY CATEGORY CODE (Standard versions):
----------------------------------------------------------------------------------------------------

3. TOP 10 TICKERS COMPARISON:
----------------------------------------------------------------------------------------------------
Ticker Std Valid % HyDE2 Valid % Difference
   AAA        4.5%         72.7%     +68.2%
   ACB        3.0%         42.2%     +39.1%
   ARM        0.0%         39.4%     +39.4%
   C32        2.5%         63.1%     +60.6%
   CAG        0.0%         51.5%     +51.5%
   CCR        2.5%         55.6%     +53.0%
   CDN        0.0%         39.6%     +39.6%
   CLL        2.0%         47.0%     +44.9%
   CNG        2.5%         69.4%     +66.9%
   DHG 

## 17. Export EDC HyDE2 Results

In [12]:
if hyde2_edc is not None and len(hyde2_edc) > 0:
    output_dir = Path('/media/nvme0n1/dev/annual_report/data/output')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 100)
    print("EXPORTING EDC HyDE2 RESULTS")
    print("=" * 100)
    
    # Export full EDC HyDE2 results
    csv_path = output_dir / 'edc_hyde2_results.csv'
    hyde2_edc.to_csv(csv_path, index=False)
    print(f"\n✓ Exported EDC HyDE2 to CSV: {csv_path}")
    print(f"  Size: {csv_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to JSON
    json_path = output_dir / 'edc_hyde2_results.json'
    hyde2_edc.to_json(json_path, orient='records', indent=2)
    print(f"✓ Exported EDC HyDE2 to JSON: {json_path}")
    print(f"  Size: {json_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to Parquet
    parquet_path = output_dir / 'edc_hyde2_results.parquet'
    hyde2_edc.to_parquet(parquet_path, index=False)
    print(f"✓ Exported EDC HyDE2 to Parquet: {parquet_path}")
    print(f"  Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export by version (standard, alt, alt_two)
    for version in ['standard', 'alt', 'alt_two']:
        version_data = hyde2_edc[hyde2_edc['version'] == version]
        if len(version_data) > 0:
            version_csv = output_dir / f'edc_hyde2_{version}_results.csv'
            version_data.to_csv(version_csv, index=False)
            print(f"✓ Exported EDC HyDE2 {version}: {version_csv} ({len(version_data):,} rows)")
    
    # Export comparison summary
    if all_results is not None:
        edc_codes = ['CC1', 'CC2', 'GHG1', 'GHG2', 'GHG3', 'GHG4', 'GHG5', 'GHG6', 'GHG7', 
                     'EC1', 'EC2', 'EC3', 'RC1', 'RC2', 'RC3', 'RC4', 'ACC1', 'ACC2']
        standard_edc = all_results[all_results['category_code'].isin(edc_codes)]
        
        comparison_summary = pd.DataFrame({
            'Metric': [
                'Total Results',
                'Valid Results',
                'Invalid Results', 
                'Valid Percentage',
                'Unique Tickers',
                'Unique Years',
                'Unique Categories'
            ],
            'Standard': [
                len(standard_edc),
                standard_edc['is_valid'].sum(),
                (~standard_edc['is_valid']).sum(),
                f"{(standard_edc['is_valid'].sum() / len(standard_edc) * 100):.1f}%",
                standard_edc['ticker'].nunique(),
                standard_edc['year'].nunique(),
                standard_edc['category_code'].nunique()
            ],
            'HyDE2': [
                len(hyde2_edc),
                hyde2_edc['is_valid'].sum(),
                (~hyde2_edc['is_valid']).sum(),
                f"{(hyde2_edc['is_valid'].sum() / len(hyde2_edc) * 100):.1f}%",
                hyde2_edc['ticker'].nunique(),
                hyde2_edc['year'].nunique(),
                hyde2_edc['category_code'].nunique()
            ]
        })
        
        comp_path = output_dir / 'edc_standard_vs_hyde2_comparison.csv'
        comparison_summary.to_csv(comp_path, index=False)
        print(f"\n✓ Exported comparison summary: {comp_path}")
    
    print(f"\n✓ All EDC HyDE2 exports completed to: {output_dir}")
else:
    print("No EDC HyDE2 results to export")

EXPORTING EDC HyDE2 RESULTS

✓ Exported EDC HyDE2 to CSV: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_results.csv
  Size: 21.09 MB
✓ Exported EDC HyDE2 to JSON: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_results.json
  Size: 26.34 MB
✓ Exported EDC HyDE2 to Parquet: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_results.parquet
  Size: 8.27 MB


/tmp/ipykernel_59971/3567366165.py:17: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  hyde2_edc.to_json(json_path, orient='records', indent=2)


✓ Exported EDC HyDE2 alt: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_alt_results.csv (16,686 rows)
✓ Exported EDC HyDE2 alt_two: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_alt_two_results.csv (16,686 rows)

✓ Exported comparison summary: /media/nvme0n1/dev/annual_report/data/output/edc_standard_vs_hyde2_comparison.csv

✓ All EDC HyDE2 exports completed to: /media/nvme0n1/dev/annual_report/data/output


## 18. Export EDC HyDE2 Results with Binary is_valid (1/0)

In [11]:
if hyde2_edc is not None and len(hyde2_edc) > 0:
    output_dir = Path('/media/nvme0n1/dev/annual_report/data/output')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 100)
    print("EXPORTING EDC HyDE2 RESULTS WITH BINARY is_valid (1/0)")
    print("=" * 100)
    
    # Convert is_valid to binary (1/0)
    hyde2_edc_binary = hyde2_edc.copy()
    hyde2_edc_binary['is_valid'] = hyde2_edc_binary['is_valid'].astype(int)
    
    # Create version column
    hyde2_edc_binary['version'] = hyde2_edc_binary['category_code'].apply(
        lambda x: 'alt_two' if '_alt_two' in x else ('alt' if '_alt' in x else 'standard')
    )
    
    # Reorder columns: ticker, year, category_code, is_valid, reason, model, created_at, top_chunks, similarities, version
    column_order = ['ticker', 'year', 'category_code', 'is_valid', 'reason', 'model', 'created_at', 'top_chunks', 'similarities', 'version']
    hyde2_edc_binary = hyde2_edc_binary[column_order]
    
    print(f"\nDataframe transformed:")
    print(f"  Columns: {list(hyde2_edc_binary.columns)}")
    print(f"  Sample rows:")
    print(hyde2_edc_binary.head(10).to_string())
    
    # Export full EDC HyDE2 results with binary is_valid
    csv_path = output_dir / 'edc_hyde2_results_binary.csv'
    hyde2_edc_binary.to_csv(csv_path, index=False)
    print(f"\n✓ Exported to CSV: {csv_path}")
    print(f"  Size: {csv_path.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"  Rows: {len(hyde2_edc_binary):,}")
    
    # Export to JSON
    json_path = output_dir / 'edc_hyde2_results_binary.json'
    hyde2_edc_binary.to_json(json_path, orient='records', indent=2, date_format='iso')
    print(f"✓ Exported to JSON: {json_path}")
    print(f"  Size: {json_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to Parquet
    parquet_path = output_dir / 'edc_hyde2_results_binary.parquet'
    hyde2_edc_binary.to_parquet(parquet_path, index=False)
    print(f"✓ Exported to Parquet: {parquet_path}")
    print(f"  Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export by version with binary is_valid
    print(f"\n✓ Exporting by version:")
    for version in ['alt', 'alt_two']:
        version_data = hyde2_edc_binary[hyde2_edc_binary['version'] == version]
        if len(version_data) > 0:
            version_csv = output_dir / f'edc_hyde2_{version}_binary.csv'
            version_data.to_csv(version_csv, index=False)
            print(f"  - EDC HyDE2 {version}: {version_csv} ({len(version_data):,} rows)")
    
    print(f"\n✓ All binary exports completed to: {output_dir}")
    print(f"\nNote: is_valid values are now: 1 (valid), 0 (invalid)")
    
else:
    print("No EDC HyDE2 results to export")

EXPORTING EDC HyDE2 RESULTS WITH BINARY is_valid (1/0)

Dataframe transformed:
  Columns: ['ticker', 'year', 'category_code', 'is_valid', 'reason', 'model', 'created_at', 'top_chunks', 'similarities', 'version']
  Sample rows:
  ticker  year category_code  is_valid                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     reason         model                 created_at                                                                                top_chunks                                                                                                 

## 19. Export ALL Inference Results with Binary is_valid (1/0)

In [14]:
if all_results is not None and len(all_results) > 0:
    output_dir = Path('/media/nvme0n1/dev/annual_report/data/output')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 100)
    print("EXPORTING ALL INFERENCE RESULTS WITH BINARY is_valid (1/0)")
    print("=" * 100)
    
    # Convert is_valid to binary (1/0)
    all_results_binary = all_results.copy()
    all_results_binary['is_valid'] = all_results_binary['is_valid'].astype(int)
    
    # Reorder columns: ticker, year, category_code, is_valid, reason, model, created_at, top_chunks, similarities
    column_order = ['ticker', 'year', 'category_code', 'is_valid', 'reason', 'model', 'created_at', 'top_chunks', 'similarities']
    all_results_binary = all_results_binary[column_order]
    
    print(f"\nDataframe transformed:")
    print(f"  Columns: {list(all_results_binary.columns)}")
    print(f"  Sample rows:")
    print(all_results_binary.head(10).to_string())
    
    # Export all results with binary is_valid
    csv_path = output_dir / 'all_inference_results_binary.csv'
    all_results_binary.to_csv(csv_path, index=False)
    print(f"\n✓ Exported to CSV: {csv_path}")
    print(f"  Size: {csv_path.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"  Rows: {len(all_results_binary):,}")
    
    # Export to JSON
    json_path = output_dir / 'all_inference_results_binary.json'
    all_results_binary.to_json(json_path, orient='records', indent=2, date_format='iso')
    print(f"✓ Exported to JSON: {json_path}")
    print(f"  Size: {json_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to Parquet
    parquet_path = output_dir / 'all_inference_results_binary.parquet'
    all_results_binary.to_parquet(parquet_path, index=False)
    print(f"✓ Exported to Parquet: {parquet_path}")
    print(f"  Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    print(f"\n✓ All binary exports completed to: {output_dir}")
    print(f"\nNote: is_valid values are now: 1 (valid), 0 (invalid)")
    
else:
    print("No inference results to export")

EXPORTING ALL INFERENCE RESULTS WITH BINARY is_valid (1/0)

Dataframe transformed:
  Columns: ['ticker', 'year', 'category_code', 'is_valid', 'reason', 'model', 'created_at', 'top_chunks', 'similarities']
  Sample rows:
  ticker  year category_code  is_valid                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     reason         model                 created_at                                                                                top_chunks                                                                                                        

## 20. Pivot EDC HyDE2 Results - One Row per Ticker-Year

In [12]:
if hyde2_edc_binary is not None and len(hyde2_edc_binary) > 0:
    output_dir = Path('/media/nvme0n1/dev/annual_report/data/output')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 100)
    print("PIVOT EDC HyDE2 RESULTS BY MODE - ONE ROW PER TICKER-YEAR")
    print("=" * 100)
    
    # Split into three modes
    hyde2_strict = hyde2_edc_binary[~hyde2_edc_binary['category_code'].str.contains('_alt', regex=True)].copy()
    hyde2_alt = hyde2_edc_binary[hyde2_edc_binary['category_code'].str.contains('_alt_two', regex=False) & 
                                 hyde2_edc_binary['category_code'].str.contains('_alt', regex=True)].copy()
    hyde2_alt_two = hyde2_edc_binary[hyde2_edc_binary['category_code'].str.contains('_alt_two', regex=True)].copy()
    
    print(f"\n✓ Data split by mode:")
    print(f"  Strict EDC: {len(hyde2_strict):,} results")
    print(f"  Alternative (_alt): {len(hyde2_alt):,} results")
    print(f"  Alternative 2 (_alt_two): {len(hyde2_alt_two):,} results")
    
    # Process each mode
    modes = [
        ('strict', hyde2_strict, 'EDC (Strict)'),
        ('alt', hyde2_alt, 'EDC (Alternative)'),
        ('alt_two', hyde2_alt_two, 'EDC (Alternative 2)')
    ]
    
    for mode_key, mode_data, mode_label in modes:
        if len(mode_data) > 0:
            print(f"\n--- {mode_label} ---")
            
            # Pivot: ticker, year as index, category_code as columns, is_valid as values
            mode_pivoted = mode_data.pivot_table(
                index=['ticker', 'year'],
                columns='category_code',
                values='is_valid',
                aggfunc='first'
            ).reset_index()
            
            print(f"Pivoted shape: {mode_pivoted.shape}")
            print(f"  Rows (ticker-year combinations): {len(mode_pivoted):,}")
            print(f"  Columns: {len(mode_pivoted.columns)}")
            
            # Export to CSV
            csv_path = output_dir / f'edc_hyde2_{mode_key}_pivoted.csv'
            mode_pivoted.to_csv(csv_path, index=False)
            print(f"✓ Exported to CSV: {csv_path}")
            print(f"  Size: {csv_path.stat().st_size / 1024 / 1024:.2f} MB")
            
            # Export to JSON
            json_path = output_dir / f'edc_hyde2_{mode_key}_pivoted.json'
            mode_pivoted.to_json(json_path, orient='records', indent=2)
            print(f"✓ Exported to JSON: {json_path}")
            print(f"  Size: {json_path.stat().st_size / 1024 / 1024:.2f} MB")
            
            # Export to Parquet
            parquet_path = output_dir / f'edc_hyde2_{mode_key}_pivoted.parquet'
            mode_pivoted.to_parquet(parquet_path, index=False)
            print(f"✓ Exported to Parquet: {parquet_path}")
            print(f"  Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    print(f"\n✓ All mode-specific pivoted exports completed to: {output_dir}")
    print(f"\nFiles created:")
    print(f"  - edc_hyde2_strict_pivoted.* (Standard EDC)")
    print(f"  - edc_hyde2_alt_pivoted.* (Alternative EDC)")
    print(f"  - edc_hyde2_alt_two_pivoted.* (Alternative 2 EDC)")
    print(f"\nNote: Each row = 1 ticker-year combination")
    print(f"      Each category_code is a column with values 0 (invalid) or 1 (valid)")
    
else:
    print("No EDC HyDE2 binary results to pivot")

PIVOT EDC HyDE2 RESULTS BY MODE - ONE ROW PER TICKER-YEAR

✓ Data split by mode:
  Strict EDC: 0 results
  Alternative (_alt): 16,686 results
  Alternative 2 (_alt_two): 16,686 results

--- EDC (Alternative) ---
Pivoted shape: (927, 20)
  Rows (ticker-year combinations): 927
  Columns: 20
✓ Exported to CSV: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_alt_pivoted.csv
  Size: 0.04 MB
✓ Exported to JSON: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_alt_pivoted.json
  Size: 0.38 MB
✓ Exported to Parquet: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_alt_pivoted.parquet
  Size: 0.01 MB

--- EDC (Alternative 2) ---
Pivoted shape: (927, 20)
  Rows (ticker-year combinations): 927
  Columns: 20
✓ Exported to CSV: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_alt_two_pivoted.csv
  Size: 0.04 MB
✓ Exported to JSON: /media/nvme0n1/dev/annual_report/data/output/edc_hyde2_alt_two_pivoted.json
  Size: 0.38 MB
✓ Exported to Parquet: /media/nvme0n1/dev/annual_repo

## 21. Pivot ALL Inference Results - One Row per Ticker-Year

In [16]:
if all_results_binary is not None and len(all_results_binary) > 0:
    output_dir = Path('/media/nvme0n1/dev/annual_report/data/output')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 100)
    print("PIVOT ALL INFERENCE RESULTS - ONE ROW PER TICKER-YEAR")
    print("=" * 100)
    
    # Pivot: ticker, year as index, category_code as columns, is_valid as values
    all_pivoted = all_results_binary.pivot_table(
        index=['ticker', 'year'],
        columns='category_code',
        values='is_valid',
        aggfunc='first'  # Take first value if duplicates exist
    ).reset_index()
    
    print(f"\n✓ Pivoted data created:")
    print(f"  Shape: {all_pivoted.shape}")
    print(f"  Rows (ticker-year combinations): {len(all_pivoted):,}")
    print(f"  Columns: {len(all_pivoted.columns)}")
    print(f"\nFirst 15 rows:")
    print(all_pivoted.head(15).to_string())
    
    # Export pivoted all results
    csv_path = output_dir / 'all_inference_results_pivoted.csv'
    all_pivoted.to_csv(csv_path, index=False)
    print(f"\n✓ Exported pivoted results to CSV: {csv_path}")
    print(f"  Size: {csv_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to JSON
    json_path = output_dir / 'all_inference_results_pivoted.json'
    all_pivoted.to_json(json_path, orient='records', indent=2)
    print(f"✓ Exported pivoted results to JSON: {json_path}")
    print(f"  Size: {json_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Export to Parquet
    parquet_path = output_dir / 'all_inference_results_pivoted.parquet'
    all_pivoted.to_parquet(parquet_path, index=False)
    print(f"✓ Exported pivoted results to Parquet: {parquet_path}")
    print(f"  Size: {parquet_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    print(f"\n✓ Pivoted exports completed to: {output_dir}")
    print(f"\nNote: Each row = 1 ticker-year combination")
    print(f"      Each category_code is a column with values 0 (invalid) or 1 (valid)")
    
else:
    print("No inference results binary data to pivot")

PIVOT ALL INFERENCE RESULTS - ONE ROW PER TICKER-YEAR

✓ Pivoted data created:
  Shape: (927, 56)
  Rows (ticker-year combinations): 927
  Columns: 56

First 15 rows:
category_code ticker  year  ACC1  ACC1_alt  ACC1_alt_two  ACC2  ACC2_alt  ACC2_alt_two  CC1  CC1_alt  CC1_alt_two  CC2  CC2_alt  CC2_alt_two  EC1  EC1_alt  EC1_alt_two  EC2  EC2_alt  EC2_alt_two  EC3  EC3_alt  EC3_alt_two  GHG1  GHG1_alt  GHG1_alt_two  GHG2  GHG2_alt  GHG2_alt_two  GHG3  GHG3_alt  GHG3_alt_two  GHG4  GHG4_alt  GHG4_alt_two  GHG5  GHG5_alt  GHG5_alt_two  GHG6  GHG6_alt  GHG6_alt_two  GHG7  GHG7_alt  GHG7_alt_two  RC1  RC1_alt  RC1_alt_two  RC2  RC2_alt  RC2_alt_two  RC3  RC3_alt  RC3_alt_two  RC4  RC4_alt  RC4_alt_two
0                AAA  2015     0         0             1     0         0             0    0        1            1    0        1            1    0        1            1    0        0            0    0        1            1     0         1             0     0         1             0     0      